# Maya v0.13C3.5 — Word Composition Diagnostic

This notebook does **not train** anything.

It compares **v0.13C3 step 750 vs step 1000** on:
- controlled contrast pseudo-words
- real Hindi words supported by the current C3 phone inventory

## Upload exactly 2 ZIPs

1. `maya_native_hindi_v013c3_checkpoint.zip`
2. `Maya_Phonetic_Foundation_v0.13C1_2_PHONE_ALIGNED.zip`

No Husn full phrase yet. `इश्क़` is intentionally excluded because the current C3 foundation does not contain `z_q`.


In [ ]:
# 1. GPU + upload packages
import subprocess
subprocess.run(["nvidia-smi"],check=True)

from google.colab import files
from pathlib import Path
import zipfile,shutil,re

u=files.upload()
zips=[Path(n) for n in u if n.lower().endswith(".zip")]
assert len(zips)==2,zips

root=Path("/content/maya_c35")
if root.exists(): shutil.rmtree(root)
root.mkdir()

packs=[]
for z in zips:
    d=root/z.stem
    d.mkdir()
    with zipfile.ZipFile(z) as zz:
        zz.extractall(d)
    packs.append((z.name,d))

checkpoint_dir=aligned_dir=None

for name,d in packs:
    if list(d.rglob("*.ckpt")):
        checkpoint_dir=d
    if list(d.rglob("raw/transcriptions.csv")) and list(d.rglob("dictionary.txt")):
        aligned_dir=d

assert checkpoint_dir,"Checkpoint ZIP not identified"
assert aligned_dir,"Aligned corpus ZIP not identified"

print("Checkpoint:",checkpoint_dir)
print("Aligned:",aligned_dir)


In [ ]:
# 2. Locate step 750 and step 1000 checkpoints + config + dictionary
from pathlib import Path
import re

ckpts=list(checkpoint_dir.rglob("*.ckpt"))
configs=list(checkpoint_dir.rglob("config.yaml"))
assert ckpts and configs

def step_of(p):
    m=re.search(r"(?:steps[_-]?|step[_-]?)(\\d+)",p.name,re.I)
    if m:return int(m.group(1))
    nums=[int(x) for x in re.findall(r"\\d+",p.stem)]
    return max(nums) if nums else -1

for p in sorted(ckpts,key=step_of):
    print(step_of(p),p)

selected={}
for step in [750,1000]:
    hits=[p for p in ckpts if step_of(p)==step]
    assert len(hits)==1,(step,hits)
    selected[step]=hits[0]

base_cfg=configs[0]

dict_candidates=list(checkpoint_dir.rglob("dictionary-mv.txt"))+list(checkpoint_dir.rglob("dictionary.txt"))
if dict_candidates:
    base_dict=dict_candidates[0]
else:
    xs=list(aligned_dir.rglob("dictionary.txt"))
    assert len(xs)==1,xs
    base_dict=xs[0]

print("Selected:",selected)
print("Config:",base_cfg)
print("Dictionary:",base_dict)


In [ ]:
# 3. Install DiffSinger
from pathlib import Path
import subprocess,shutil,sys

repo=Path("/content/DiffSinger")
venv=Path("/content/diffsinger-env")

for p in [repo,venv]:
    if p.exists(): shutil.rmtree(p)

subprocess.run(["git","clone","https://github.com/openvpi/DiffSinger.git",str(repo)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
subprocess.run(["uv","python","install","3.10"],check=True)
subprocess.run(["uv","venv","--python","3.10","--seed",str(venv)],check=True)

PY="/content/diffsinger-env/bin/python"
subprocess.run([PY,"-m","pip","install","-U","pip","wheel"],check=True)
subprocess.run([PY,"-m","pip","install","--force-reinstall","setuptools==81.0.0"],check=True)
subprocess.run([
    PY,"-m","pip","install",
    "torch==2.4.0","torchaudio==2.4.0","torchvision==0.19.0",
    "--index-url","https://download.pytorch.org/whl/cu121"
],check=True)
subprocess.run([PY,"-m","pip","install","-r",str(repo/"requirements.txt")],check=True)
subprocess.run([PY,"-m","pip","install","--force-reinstall","setuptools==81.0.0"],check=True)


In [ ]:
# 4. Install checkpoint experiments + vocoder
from pathlib import Path
import shutil,json,yaml,requests,zipfile,io

REPO=Path("/content/DiffSinger")

for step,ckpt in selected.items():
    exp=REPO/"checkpoints"/f"maya_c3_step_{step}"
    exp.mkdir(parents=True,exist_ok=True)

    shutil.copy2(ckpt,exp/f"model_ckpt_steps_{step}.ckpt")
    shutil.copy2(base_cfg,exp/"config.yaml")
    shutil.copy2(base_dict,exp/"dictionary-mv.txt")

    cfg=yaml.safe_load((exp/"config.yaml").read_text(encoding="utf-8"))

    if not (exp/"lang_map.json").exists():
        (exp/"lang_map.json").write_text(
            json.dumps({"mv":0},indent=2),
            encoding="utf-8"
        )

    if cfg.get("use_spk_id",False) and not (exp/"spk_map.json").exists():
        (exp/"spk_map.json").write_text(
            json.dumps({"maya":0},indent=2),
            encoding="utf-8"
        )

url=(
"https://github.com/openvpi/vocoders/releases/download/"
"pc-nsf-hifigan-44.1k-hop512-128bin-2025.02/"
"pc_nsf_hifigan_44.1k_hop512_128bin_2025.02.zip"
)

r=requests.get(url,timeout=180)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall(REPO/"checkpoints")

print("Experiments ready.")


In [ ]:
# 5. Build controlled contrast and real-word tests
from pathlib import Path
import json,numpy as np

REPO=Path("/content/DiffSinger")
tests=REPO/"data"/"maya_c35_tests"
tests.mkdir(parents=True,exist_ok=True)

phones={"AP","SP"}

for line in base_dict.read_text(encoding="utf-8").splitlines():
    if line.strip():
        _,rhs=line.split("\\t",1)
        phones.update(rhs.split())

print("Phone inventory:",sorted(phones))

def make_ds(name,seq,durs,freq=235.0):
    missing=[p for p in seq if p!="SP" and p not in phones]
    assert not missing,(name,missing)

    duration=float(sum(durs))
    dt=.01
    t=np.arange(0,duration,dt)

    midi=69+12*np.log2(freq/440.0)+.08*np.sin(2*np.pi*5*t)
    f0=440*2**((midi-69)/12)

    ds=[{
        "offset":0.0,
        "ph_seq":" ".join(seq),
        "ph_dur":" ".join(f"{x:.6f}" for x in durs),
        "f0_seq":" ".join(f"{x:.4f}" for x in f0),
        "f0_timestep":dt,
        "lang":"mv"
    }]

    p=tests/f"{name}.ds"
    p.write_text(json.dumps(ds,ensure_ascii=False,indent=2),encoding="utf-8")
    return p

controlled=[
 ("01_kaana",["z_k","a","z_n","a"]),
 ("02_khaana",["z_kh","a","z_n","a"]),
 ("03_chaana",["z_ch","a","z_n","a"]),
 ("04_chhaana",["z_chh","a","z_n","a"]),
 ("05_vaana",["z_v","a","z_n","a"]),
 ("06_yaana",["z_y","a","z_n","a"]),
 ("07_maana",["z_m","a","z_n","a"]),
 ("08_naana",["z_n","a","z_n","a"]),
 ("09_taana",["z_t","a","z_n","a"]),
 ("10_thaana",["z_th","a","z_n","a"]),
 ("11_daana",["z_d","a","z_n","a"]),
 ("12_dhaana",["z_dh","a","z_n","a"]),
 ("13_paana",["z_p","a","z_n","a"]),
 ("14_phaana",["z_ph","a","z_n","a"]),
 ("15_jaana",["z_j","a","z_n","a"]),
 ("16_jhaana",["z_jh","a","z_n","a"]),
 ("17_gaana",["z_g","a","z_n","a"]),
 ("18_ghaana",["z_gh","a","z_n","a"])
]

controlled_tests=[]
for name,seq in controlled:
    durs=[.24 if p.startswith("z_") else .52 for p in seq]
    controlled_tests.append(make_ds(name,seq,durs,230.0))

real=[
 ("20_husn",["z_h","u","z_s","z_n"]),
 ("21_suhaana",["z_s","u","z_h","a","z_n","a"]),
 ("22_deewaana",["z_d","i","z_v","a","z_n","a"]),
 ("23_roop",["z_r","u","z_p"]),
 ("24_khajaana",["z_kh","a","z_j","a","z_n","a"]),
 ("25_lutaana",["z_l","u","z_T","a","z_n","a"]),
 ("26_seene",["z_s","i","z_n","e"]),
 ("27_lagaana",["z_l","a","z_g","a","z_n","a"])
]

real_tests=[]
for name,seq in real:
    durs=[.24 if p.startswith("z_") else .55 for p in seq]
    real_tests.append(make_ds(name,seq,durs,240.0))

print("Controlled:",len(controlled_tests),"Real:",len(real_tests))


In [ ]:
# 6. Render step 750 vs step 1000 and download four WAVs
import os,subprocess,shutil
from pathlib import Path
import soundfile as sf,numpy as np
from google.colab import files
from IPython.display import Audio,display

PY="/content/diffsinger-env/bin/python"
REPO=Path("/content/DiffSinger")

env=os.environ.copy()
env["MPLBACKEND"]="Agg"
env["PYTHONUNBUFFERED"]="1"

def montage(paths,out):
    parts=[]
    sr0=None

    for p in paths:
        y,sr=sf.read(p)

        if y.ndim>1:
            y=y.mean(axis=1)

        sr0=sr0 or sr

        peak=np.max(np.abs(y))+1e-9
        if peak>.9:
            y*=.9/peak

        parts += [y,np.zeros(int(.65*sr))]

    sf.write(out,np.concatenate(parts),sr0)
    display(Audio(str(out)))

outs=[]

for step in [750,1000]:
    exp=f"maya_c3_step_{step}"
    outroot=Path(f"/content/maya_c35_step_{step}")

    if outroot.exists():
        shutil.rmtree(outroot)

    outroot.mkdir()

    for ds in controlled_tests+real_tests:
        p=subprocess.run(
            [
                PY,"scripts/infer.py","acoustic",
                str(ds.relative_to(REPO)),
                "--exp",exp,
                "--ckpt",str(step),
                "--out",str(outroot/ds.stem)
            ],
            cwd=str(REPO),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env
        )

        if p.returncode!=0:
            print(p.stdout)
            raise RuntimeError(f"Render failed step={step} {ds.name}")

    wavs=sorted(outroot.rglob("*.wav"))

    controlled_w=[p for p in wavs if int(p.stem.split("_")[0])<=18]
    real_w=[p for p in wavs if int(p.stem.split("_")[0])>=20]

    assert len(controlled_w)==18
    assert len(real_w)==8

    pc=Path(f"/content/Maya_v013C35_step{step}_CONTROLLED_WORDS.wav")
    pr=Path(f"/content/Maya_v013C35_step{step}_REAL_WORDS.wav")

    print("STEP",step,"CONTROLLED")
    montage(controlled_w,pc)

    print("STEP",step,"REAL")
    montage(real_w,pr)

    outs += [pc,pr]

for p in outs:
    files.download(str(p))

print("EXPORT COMPLETE")


Upload these four WAVs back here:

- `Maya_v013C35_step750_CONTROLLED_WORDS.wav`
- `Maya_v013C35_step750_REAL_WORDS.wav`
- `Maya_v013C35_step1000_CONTROLLED_WORDS.wav`
- `Maya_v013C35_step1000_REAL_WORDS.wav`

No training happens here. We choose the checkpoint by actual word intelligibility before C4.
